# WideResNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，Wide Residual Network（WideResNet, WRN）を用いてCIFAR100データセットの100クラス物体認識を行い，ネットワークの「幅」を広げることによる高精度化について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Wide Residual Network（WideResNet）
ResNetは，スキップ構造の導入によりネットワークを非常に深くすることを可能にしましたが，層数を増やすほど精度向上が頭打ちになりやすく，また各Residual Blockの計算は前の層の出力を待って逐次的に実行する必要があるため，深いネットワークほど学習・推論に時間がかかりGPUの並列計算能力を活かしきれないという問題があります．

WideResNetは，「ネットワークを深くする」代わりに「Residual Blockのチャンネル数（幅）を広げる」ことでモデルの表現能力を高めるアプローチを提案しました．幅を広げる分だけ層数を浅くできるため，同程度のパラメータ数でも層をまたぐ逐次計算の回数が減り，GPU上での並列計算の恩恵を受けやすく効率的に学習できます．また，幅を広げたモデルは過学習しやすくなるため，WideResNetではResidual Block内の2つの畳み込み層の間にDropoutを挿入し，正則化を行うことが提案されています．

### Wide Basic Block
WideResNetのResidual Blockは，ResNetのBasicBlockと同様に3×3の畳み込みを2つ用いた構造ですが，1つ目の畳み込み・BatchNorm・ReLUの後にDropoutを挿入する点が異なります．各ブロックの出力チャンネル数は，widening factor（幅の拡大率）`k`を用いて`16k`，`32k`，`64k`のように通常のResNetの`k`倍に設定します．

### CIFAR100への入力サイズの適応
WideResNetも，ベースとなるResNetと同様に，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR10/100を対象とした「CIFAR版」があります．本ノートブックでは，`resnet.ipynb`のCIFAR版ResNet（3×3のstem，特徴マップサイズごとに3ブロック群）をベースに，各ブロックのチャンネル数をwidening factorで拡大したCIFAR版のWideResNetを実装します（詳細は本ノートブック末尾の「オリジナル（ImageNet版）WideResNetとの違い」を参照）．

In [ ]:
class WideBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None, drop_rate=0.0):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Dropout(p=drop_rate),  # 幅を広げたことによる過学習を抑えるためのDropout
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.convs(x)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

### WideResNetの定義
上で定義したWide Basic Blockを用いてWideResNetを構築します．`depth`は層数，`widen_factor`は幅の拡大率`k`，`drop_rate`はDropoutの割合を指定します．

WideResNetの層数は，ResNetのCIFAR版と同様に3つのブロック群から構成されますが，論文の慣例に従い

$$\text{depth} = (\text{Res. Block内の畳み込み数}=2) \times (\text{1ブロックあたりのRes. Block数}) \times 3 + 4$$

という関係で層数を指定します（例：WRN-16-8は`depth=16`, `widen_factor=8`）．最初の畳み込み層は幅の拡大の対象外とし，チャンネル数は16のまま固定します．

In [ ]:
class WideResNet(nn.Module):
    def __init__(self, depth, widen_factor=1, drop_rate=0.0, n_class=100):
        super().__init__()
        # 指定した深さ（畳み込みの層数）でネットワークを構築できるかを確認
        assert (depth - 4) % 6 == 0, 'depth should be 6n+4 (e.g. 16, 22, 28, 40).'
        n_blocks = (depth - 4) // 6  # 1ブロックあたりのWide Basic Blockの数を決定

        self.inplanes = 16
        k = widen_factor

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16 * k, n_blocks, drop_rate=drop_rate)
        self.layer2 = self._make_layer(32 * k, n_blocks, stride=2, drop_rate=drop_rate)
        self.layer3 = self._make_layer(64 * k, n_blocks, stride=2, drop_rate=drop_rate)

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * k * WideBasicBlock.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1, drop_rate=0.0):
        downsample = None
        if stride != 1 or self.inplanes != planes * WideBasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * WideBasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * WideBasicBlock.expansion),
            )

        layers = [WideBasicBlock(self.inplanes, planes, stride, downsample, drop_rate=drop_rate)]
        self.inplanes = planes * WideBasicBlock.expansion
        for _ in range(n_blocks - 1):
            layers.append(WideBasicBlock(self.inplanes, planes, drop_rate=drop_rate))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`n_layers`で層数，`widen_factor`で幅の拡大率を指定し，`WideResNet`を呼び出します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
# WideResNetの層数と幅の拡大率を指定 (e.g. WRN-16-8, WRN-28-10)
n_layers = 16
widen_factor = 8

model = WideResNet(depth=n_layers, widen_factor=widen_factor, drop_rate=0.3, n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## オリジナル（ImageNet版）WideResNetとの違い
このノートブックで実装したWideResNetは，CIFAR10/100を想定したCIFAR版の構造です．ImageNet向けのオリジナル構造とは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 1層目の畳み込みのカーネルサイズ | 7×7 | 3×3 |
| 1層目の畳み込み後のチャンネル数 | 64 | 16 |
| 特徴マップサイズごとのブロック数 | 4 | 3 |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

また，通常のCIFAR版ResNet（`resnet.ipynb`）と比較すると，WideResNet固有の変更点は次の通りです．

| | CIFAR版ResNet | WideResNet（本ノートブック） |
|---|---|---|
| 各ブロック群のチャンネル数 | 16, 32, 64 | 16k, 32k, 64k（`k`=widen_factor） |
| Residual Block内のDropout | なし | 2つの畳み込みの間に挿入 |
| 層数の数え方 | depth = 6n+2 | depth = 6n+4 |

## 課題

1. `widen_factor`（例：1, 4, 8）を変更して，パラメータ数や認識精度がどのように変化するか確認しましょう．

2. 同程度のパラメータ数となるように`resnet.ipynb`の（より層数の深い）ResNetを構築し，「深さ」と「幅」のどちらがCIFAR100の認識精度に効果的か比較しましょう．

3. `drop_rate`を0（Dropoutなし）に変更して学習し，Dropoutの有無による学習の推移や認識精度の違いを確認しましょう．